# 01 — Data: read, normalize, describe, explore

**Part A (cells tagged `lib`)** is the data layer the agent uses; `02_agent` and `03_evaluate` load
it with `run_lib_cells("01_data.ipynb", globals())` without re-running the exploration.

1. Reading a split: questions, answers, READMEs, raw tables (csv / xlsx sheets / sqlite tables)
2. Header detection (README column names, else a heuristic)
3. Normalization: parsed `__date` / `__dt` / `__num` columns, fill-down, `ALL …*` combined views
4. `Business` + cache, and the plain pickle the sandbox reads
5. The table description shown to the model

**Part B** is EDA: questions, answers, files, messiness, rulebook drift, and a data card per shop.

In [ ]:
SPLITS = ["train", "public_test"]

In [ ]:
import datetime as dt
import json
import math
import os
import pickle
import re
import sqlite3
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

from datamate.helpers import dates, ffill_orders, fold, money, to_date, to_datetime, to_number, year_of
from datamate.sandbox import save_tables


def find_repo() -> Path:
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "pyproject.toml").exists() and (d / "notebooks").is_dir():
            return d
    return Path.cwd()


REPO = find_repo()
DATA = Path(os.environ.get("DATAMATE_DATA_ROOT", REPO / "data"))      # holds train/, public_test/
CACHE = Path(os.environ.get("DATAMATE_CACHE", DATA / "cache"))
CACHE_VERSION = 6   # bump when what load_business produces changes

## A1. Reading a split
A split is a folder with `questions.jsonl`, `rulebook.md`, `calendar.csv`, `data/<business>/` and,
for train, `answers.jsonl`. In Docker it is `/data` itself; in development `data/<name>/`.

In [ ]:
def split_root(split) -> Path:
    """'train' / 'public_test' -> DATA/<name>; a path to a split folder -> itself."""
    p = Path(split)
    return p if (p / "questions.jsonl").exists() else DATA / str(split)


def read_jsonl(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def to_json_line(obj: dict) -> str:
    return json.dumps(obj, ensure_ascii=False)


def load_questions(split) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(split_root(split) / "questions.jsonl"))


def load_answers(split) -> pd.DataFrame | None:
    path = split_root(split) / "answers.jsonl"
    return pd.DataFrame(read_jsonl(path)) if path.exists() else None


def business_ids(split) -> list[str]:
    return sorted(p.name for p in (split_root(split) / "data").iterdir() if p.is_dir())


@dataclass
class Readme:
    text: str
    files: list[str]
    columns: dict[str, dict[str, str]] = field(default_factory=dict)   # "file[, sheet X]" -> {column: meaning}
    conventions: list[str] = field(default_factory=list)
    last_date: str | None = None


_SECTION = re.compile(r"^## (.+)$", re.M)
_COL = re.compile(r"^- \*\*(.+?)\*\*: (.+)$")


def parse_readme(path: Path) -> Readme:
    text = path.read_text(encoding="utf-8")
    heads = list(_SECTION.finditer(text))
    files, columns, conventions = [], {}, []
    for i, h in enumerate(heads):
        body = text[h.end(): heads[i + 1].start() if i + 1 < len(heads) else len(text)]
        title = h.group(1).strip()
        if title == "File":
            files = re.findall(r"^- `([^`]+)`", body, re.M)
        elif title.startswith("Cột trong"):
            key = title.removeprefix("Cột trong").strip().replace("`", "")
            columns[key] = {m.group(1): m.group(2) for m in map(_COL.match, body.splitlines()) if m}
        elif title == "Quy ước":
            conventions = [ln[2:] for ln in body.splitlines() if ln.startswith("- ")]
    m = re.search(r"Dữ liệu có (?:từ [\d/]+ )?đến hết (\d{2}/\d{2}/\d{4})", text)
    return Readme(text, files, columns, conventions, m.group(1) if m else None)


@dataclass
class RawTable:
    file: str
    sheet: str | None   # xlsx sheet or sqlite table; None for csv
    df: pd.DataFrame    # header=None, every cell as read


def read_raw_tables(biz_dir: Path) -> list[RawTable]:
    """Every sheet / table of a business, without header inference."""
    out = []
    for p in sorted(biz_dir.iterdir()):
        suf = p.suffix.lower()
        if suf == ".csv":
            out.append(RawTable(p.name, None, pd.read_csv(p, header=None, dtype=str, encoding="utf-8-sig",
                                                          keep_default_na=False, na_values=[""])))
        elif suf == ".xlsx":
            for sheet, df in pd.read_excel(p, sheet_name=None, header=None).items():
                out.append(RawTable(p.name, sheet, df))
        elif suf == ".sqlite":
            con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
            for (t,) in con.execute("select name from sqlite_master where type='table'").fetchall():
                df = pd.read_sql_query(f'select * from "{t}"', con)
                out.append(RawTable(p.name, t, pd.concat([df.columns.to_frame().T, df], ignore_index=True)
                                    .set_axis(range(df.shape[1]), axis=1)))
            con.close()
    return out

## A2. Header detection
Where the README describes a file's columns, the header is the row (within the first 10) that
covers the most README column names: 100% coverage on every described table (EDA §5). Otherwise
the first row that is mostly non-empty strings, which skips group-header rows.

In [ ]:
def readme_columns_for(readme: Readme, table) -> dict[str, str]:
    """README column map of a table: matches `file`, `file, sheet X`, `pattern<...>.xlsx` or a sqlite table."""
    best = {}
    for key, cols in readme.columns.items():
        fpart, _, spart = key.partition(", sheet ")
        rx = "^" + re.sub(r"<[^>]*>", ".+", re.escape(fpart.strip())) + "$"
        if (re.match(rx, table.file) or (table.sheet and fpart.strip() == table.sheet)) and \
                (not spart or spart.strip() == table.sheet):
            best = cols
        elif not best and fpart.strip() in (table.file, table.sheet or ""):
            best = cols
    return best


def find_header_row(df: pd.DataFrame, expected: list[str], scan: int = 10) -> tuple[int, float]:
    """(row, coverage) of the row within the first `scan` that best covers the README column names."""
    if not expected:
        return 0, float("nan")
    want = {fold(c) for c in expected}
    best = (0, -1.0)
    for i in range(min(scan, len(df))):
        got = {fold(v) for v in df.iloc[i].tolist() if isinstance(v, str)}
        cov = len(want & got) / len(want)
        if cov > best[1]:
            best = (i, cov)
    return best


def guess_header_row(df: pd.DataFrame, scan: int = 8) -> int:
    ncols = df.shape[1]
    for i in range(min(scan, len(df) - 1)):
        vals = [v for v in df.iloc[i].tolist() if v is not None and not (isinstance(v, float) and pd.isna(v))]
        if len(vals) >= max(1, 0.6 * ncols) and all(isinstance(v, str) for v in vals):
            return i
    return 0


def drop_header_repeats(df: pd.DataFrame) -> pd.DataFrame:
    """Rows that repeat the header (an export glitch in some files) are not data: at least half the columns,
    and two or more, hold their own column name."""
    if df.empty:
        return df
    same = sum((df[c].astype(str).str.strip().str.lower() == str(c).strip().lower()).astype(int) for c in df.columns)
    rep = same >= max(2, len(df.columns) // 2)
    return df[~rep].reset_index(drop=True) if rep.any() else df


def with_header(df: pd.DataFrame, row: int) -> pd.DataFrame:
    cols = [str(c) if c is not None and not (isinstance(c, float) and pd.isna(c)) else f"_c{i}"
            for i, c in enumerate(df.iloc[row].tolist())]
    return df.iloc[row + 1:].set_axis(cols, axis=1).reset_index(drop=True)

## A3. Normalization
v2's validation logs showed the model ignoring the parse helpers: it called `pd.to_datetime` on
`'1 thg 6 2025'`, every date became NaT, filters kept nothing and it answered 0. So parsing is done
here, once, deterministically. Raw columns stay; we only add:
- `<col>__date` (datetime64 at midnight), `<col>__dt` when cells carry a time, `<col>__num` (float);
- fill-down of order ids and order-level cells onto follow-up lines (the files' convention, rulebook §8);
- `ALL <prefix>*` views concatenating same-column tables (one per month / part) with `__source`.

Parsing is decided per group of same-column tables, so a combined view never has a parsed column
that is empty for some months. Checked against train answers in `03_evaluate` §4: plain monthly
revenue computed by fixed code over these tables matches all 71 answers it applies to.

In [ ]:
DATE_SUFFIX, DATETIME_SUFFIX, NUM_SUFFIX, SOURCE_COL = "__date", "__dt", "__num", "__source"
GROUP_COL = "__group"
PARSE_SAMPLE = 3000
MIN_PARSE = 0.9           # share of non-blank cells a parser must read before a parsed column is added
MAX_COMBINED_ROWS = int(os.environ.get("DATAMATE_MAX_COMBINED_ROWS", 2_000_000))
_ORDER_ID_ROLE = re.compile(r"^(ma don|so hoa don|so thu tu don|ma hoa don|ma bill)\b")   # not 'so thu tu dong'
# Ids written as one word or snake case: MaHoaDon, ma_don, SoHD, OrderID.
_ORDER_ID_SQUASHED = re.compile(r"^(ma|so)(don|donhang|hoadon|hd|bill)$|^(order|invoice)(id|no|code)?$")
_WORD = re.compile(r"[^\W\d_]+")
_PHONE_LIKE = re.compile(r"\s*0\d{8,10}\s*")   # phone numbers and zero-padded codes are ids, not amounts
_DIGITS = re.compile(r"\s*-?\d+(?:\.\d+)?\s*")
_ORDER_ID_NAME = re.compile(r"^(ma|so)\s*(don|hoa don|hd|bill|phieu)|order|invoice|^stt don")


@dataclass(frozen=True)
class Parsed:
    raw: str
    col: str            # the added column
    kind: str           # "date" | "datetime" | "number"
    share: float        # share of non-blank raw cells it read
    lo: str = ""        # date range of the parsed column
    hi: str = ""


def _is_blank(v) -> bool:
    return v is None or (isinstance(v, float) and math.isnan(v)) or (isinstance(v, str) and not v.strip())


def _has_time(values) -> bool:
    return any(not pd.isna(t) and (t.hour or t.minute or t.second) for t in map(to_datetime, values))


def plan_parsing(df: pd.DataFrame, table: str) -> dict[str, tuple[str, float, bool]]:
    """{raw column: (kind 'date'|'number', share read, has time)} for columns worth a parsed copy."""
    year = year_of(table)
    plan = {}
    for c in df.columns:
        col = df[c]
        if str(c).endswith((DATE_SUFFIX, DATETIME_SUFFIX, NUM_SUFFIX, SOURCE_COL, GROUP_COL)) or col.dtype.kind == "b":
            continue
        if col.dtype.kind in "iuf":   # every number column gets a __num copy, as the prompt says
            plan[str(c)] = ("number", 1.0, False)
            continue
        vals = [v for v in col.head(20 * PARSE_SAMPLE).tolist() if not _is_blank(v)][:PARSE_SAMPLE]
        if not vals:
            continue
        n_dt = sum(isinstance(v, (dt.date, pd.Timestamp)) for v in vals)
        n_str = sum(isinstance(v, str) for v in vals)
        n_digits = sum(isinstance(v, str) and _DIGITS.fullmatch(v) is not None for v in vals)
        if n_str == 0 and n_dt == 0:
            if all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals):
                plan[str(c)] = ("number", 1.0, False)   # numbers stored as object: same uniform __num copy
            continue
        date_share = sum(not pd.isna(to_date(v, year)) for v in vals) / len(vals)
        # Ints alone never make a date column (45000 VND is also a valid Excel serial), nor does
        # digit-only text, which is how a CSV stores the same ints.
        if date_share >= MIN_PARSE and (n_dt + n_str - n_digits) / len(vals) >= 0.2:
            plan[str(c)] = ("date", round(date_share, 3), _has_time(vals[:500]))
        elif n_str and not n_dt:
            num_share = sum(not math.isnan(to_number(v)) for v in vals) / len(vals)
            words = sum(isinstance(v, str) and not any(ch.isdigit() for ch in v) for v in vals) / len(vals)
            # A number may carry a unit or currency word ('2 đĩa', '55k'), not a phrase ('93 Nguyễn Thị Minh Khai').
            texty = sum(isinstance(v, str) and len(_WORD.findall(fold(v))) > 2 for v in vals) / len(vals)
            phones = sum(isinstance(v, str) and _PHONE_LIKE.fullmatch(v) is not None for v in vals) / len(vals)
            if num_share >= MIN_PARSE and words < 0.1 and texty < 0.1 and phones < 0.5:
                plan[str(c)] = ("number", round(num_share, 3), False)
    return plan


def merge_plans(plans: list[dict]) -> dict:
    """A column is parsed in every table of a group if at least half the tables planned it."""
    votes: dict[str, list] = {}
    for p in plans:
        for c, v in p.items():
            votes.setdefault(c, []).append(v)
    merged = {}
    for c, vs in votes.items():
        kinds = [v[0] for v in vs]
        kind = max(set(kinds), key=kinds.count)
        if kinds.count(kind) * 2 >= len(plans):
            same = [v for v in vs if v[0] == kind]
            merged[c] = (kind, min(v[1] for v in same), any(v[2] for v in same))
    return merged


def apply_parsing(df: pd.DataFrame, table: str, plan: dict) -> tuple[pd.DataFrame, list[Parsed]]:
    """Add the planned parsed columns; the year for '1/4'-style dates comes from this table's name."""
    year = year_of(table)
    out, added = df.copy(), []
    for c, (kind, share, has_time) in plan.items():
        if c not in out.columns:
            continue
        if kind == "date":
            d = pd.to_datetime(dates(out[c], year), errors="coerce")
            out[c + DATE_SUFFIX] = d
            ok = d.notna().any()
            added.append(Parsed(c, c + DATE_SUFFIX, "date", share,
                                str(d.min().date()) if ok else "", str(d.max().date()) if ok else ""))
            if has_time:
                out[c + DATETIME_SUFFIX] = pd.to_datetime(out[c].map(lambda v: to_datetime(v, year)), errors="coerce")
                added.append(Parsed(c, c + DATETIME_SUFFIX, "datetime", share))
        else:
            out[c + NUM_SUFFIX] = pd.to_numeric(money(out[c]), errors="coerce")
            added.append(Parsed(c, c + NUM_SUFFIX, "number", share))
    return out, added


def order_id_column(df: pd.DataFrame, roles: dict[str, str]) -> str | None:
    """The order-id column of an order-line table: by README meaning, else by name."""
    for c, role in roles.items():
        if c in df.columns and _ORDER_ID_ROLE.match(fold(role)):
            return c
    by_name = next((c for c in df.columns if _ORDER_ID_NAME.search(fold(c))), None)
    return by_name or next((c for c in df.columns if _ORDER_ID_SQUASHED.match(re.sub(r"[^a-z0-9]", "", fold(c)))),
                           None)


def fill_down(df: pd.DataFrame, roles: dict[str, str]) -> tuple[pd.DataFrame, str | None]:
    """Fill order ids and order-level cells down when follow-up lines leave them blank."""
    c = order_id_column(df, roles)
    if c is None or len(df) == 0:
        return df, None
    blank = df[c].map(_is_blank)
    # Only when blank ids really mark follow-up lines: some blanks, and the table starts with an id.
    if not (0.02 <= blank.mean() <= 0.95) or blank.iloc[0]:
        return df, None
    return ffill_orders(df, c), c


def _blank_mask(s: pd.Series) -> pd.Series:
    return s.isna() | s.map(lambda v: isinstance(v, str) and not v.strip())


def _date_like(s: pd.Series, year) -> bool:
    vals = [v for v in s.head(20 * PARSE_SAMPLE).tolist() if not _is_blank(v)][:500]
    if not vals or all(isinstance(v, (int, float)) for v in vals):
        return False
    return sum(not pd.isna(to_date(v, year)) for v in vals) / len(vals) >= MIN_PARSE


def continuation_rows(df: pd.DataFrame, table: str) -> tuple[pd.Series | None, list[str]]:
    """Rows that continue the record above in a book with no id on every line: two or more columns, a
    date among them, blank on exactly the same rows while the first row is filled. A blank customer
    code and phone alone is a walk-in customer, not a continuation, hence the date."""
    for c in df.columns:   # a column filled and distinct on every line: each line is its own record
        s = df[c]
        if len(s) and not _blank_mask(s).any() and s.nunique() >= 0.95 * len(s):
            return None, []
    masks: dict[bytes, list[str]] = {}
    for c in df.columns:
        b = _blank_mask(df[c])
        if len(b) and 0.2 <= b.mean() <= 0.95 and not b.iloc[0]:
            masks.setdefault(b.to_numpy().tobytes(), []).append(c)
    year = year_of(table)
    for cols in sorted(masks.values(), key=len, reverse=True):
        if len(cols) >= 2 and any(_date_like(df[c], year) for c in cols):
            return _blank_mask(df[cols[0]]), cols
    return None, []


def _is_amount(df: pd.DataFrame, c) -> bool:
    """Money / quantity columns: numeric dtype, a parsed __num copy, or object cells that are numbers."""
    if df[c].dtype.kind in "iufb" or str(c).endswith(NUM_SUFFIX) or f"{c}{NUM_SUFFIX}" in df.columns:
        return True
    vals = df[c].dropna().head(PARSE_SAMPLE)
    return len(vals) > 0 and vals.map(lambda v: isinstance(v, (int, float)) and not isinstance(v, bool)).mean() >= 0.9


def fill_within_records(df: pd.DataFrame, key: str) -> list[str]:
    """Rulebook §8: a cell left blank on a later line of a record equals that cell on the record's first line.
    Only columns whose non-blank values agree within (almost) every record are record-level; a value written
    on a later line only (a note on one item) stays there. Amounts are never copied: a copied amount would be
    counted once per line."""
    key_s = df[key]
    first = ~key_s.duplicated() & key_s.notna()
    multi = key_s.map(key_s.value_counts()) > 1
    filled = []
    if not multi.any():
        return filled
    for c in df.columns:
        if c == key or _is_amount(df, c):
            continue
        col = df[c].where(~_blank_mask(df[c]))
        if not col.isna().any() or col.notna().sum() == 0:
            continue
        if (col[multi].groupby(key_s[multi]).nunique() > 1).mean() > 0.01:
            continue
        head = col.where(first).groupby(key_s).transform("first")
        todo = col.isna() & head.notna()
        if todo.any():
            df[c] = df[c].where(~todo, head)
            filled.append(c)
    return filled


def dedupe_order_table(df: pd.DataFrame, roles: dict[str, str]) -> tuple[pd.DataFrame, int]:
    """Rulebook §8: two lines of the same order identical in every cell are an export error and count once.
    Lines of different orders differ in the order id, so they stay. Only tables keyed by an order id whose
    values repeat (order lines); a table not kept by order keeps identical lines. Run after the fill-down,
    so a follow-up line is compared with its order's values filled in."""
    c = order_id_column(df, roles)
    if c is None or not df[c].duplicated().any():
        return df, 0
    dup = df.duplicated()
    return (df[~dup].reset_index(drop=True), int(dup.sum())) if dup.any() else (df, 0)


def fill_records(df: pd.DataFrame, table: str, roles: dict[str, str]) -> tuple[pd.DataFrame, str | None]:
    """After parsing (so dates compare by value, not spelling): fill record-level cells left blank on some
    lines. The record key is the order id when every line has one, else `__group` from continuation_rows."""
    c = order_id_column(df, roles)
    if c is not None and len(df) and _blank_mask(df[c]).mean() <= 0.01:   # a footer line may lack the id
        out = df.copy()
        return (out, c) if fill_within_records(out, c) else (df, None)
    cont, _ = continuation_rows(df, table)
    if cont is None:
        return df, None
    out = df.copy()
    out[GROUP_COL] = (~cont).cumsum()
    fill_within_records(out, GROUP_COL)
    return out, GROUP_COL

## A4. Business and cache
`load_business` = raw tables → header → fill-down → parsed columns → combined views. Cached per
split folder and business; `sandbox_tables` writes the plain pickle the code sandbox reads.

In [ ]:
@dataclass
class Business:
    biz_id: str
    readme: Readme
    frames: dict[str, pd.DataFrame]                                  # table key -> DataFrame
    roles: dict[str, dict[str, str]] = field(default_factory=dict)  # table -> {column: README meaning}
    parsed: dict[str, list[Parsed]] = field(default_factory=dict)   # table -> parsed columns added
    filled: dict[str, str] = field(default_factory=dict)            # table -> order-id column filled down
    combined: dict[str, list[str]] = field(default_factory=dict)    # "ALL …*" key -> its source tables


def table_key(t: RawTable, multi_sheet: bool) -> str:
    if t.file.endswith(".sqlite"):
        return t.sheet
    return f"{t.file}::{t.sheet}" if multi_sheet and t.sheet else t.file


def load_business(biz_dir: Path) -> Business:
    readme = parse_readme(biz_dir / "README.md")
    raw = read_raw_tables(biz_dir)
    sheets = Counter(t.file for t in raw)
    biz = Business(biz_dir.name, readme, {})
    for t in raw:
        roles = {} if t.file.endswith(".sqlite") else readme_columns_for(readme, t)
        row = 0   # sqlite: read_raw_tables put the real column names on row 0
        if not t.file.endswith(".sqlite"):
            row, cov = find_header_row(t.df, list(roles))
            if not roles or not cov:
                row, roles = guess_header_row(t.df), {}
        key = table_key(t, sheets[t.file] > 1)
        df = with_header(t.df, row).dropna(how="all").dropna(axis=1, how="all").reset_index(drop=True)
        biz.frames[key] = drop_header_repeats(df)
        biz.roles[key] = roles
    return normalize(biz)


def normalize(biz: Business) -> Business:
    for n, df in list(biz.frames.items()):
        biz.frames[n], col = fill_down(df, biz.roles.get(n, {}))
        if col:
            biz.filled[n] = col
    groups: dict[tuple, list[str]] = {}
    for n, df in biz.frames.items():
        groups.setdefault(tuple(map(str, df.columns)), []).append(n)
    for names in groups.values():
        plans = [plan_parsing(biz.frames[n], n) for n in names]
        plan = merge_plans(plans) if len(names) > 1 else plans[0]
        for n in names:
            biz.frames[n], biz.parsed[n] = apply_parsing(biz.frames[n], n, plan)
            if n not in biz.filled:
                biz.frames[n], key = fill_records(biz.frames[n], n, biz.roles.get(n, {}))
                if key:
                    biz.filled[n] = key
            biz.frames[n], _ = dedupe_order_table(biz.frames[n], biz.roles.get(n, {}))
        if len(names) > 1 and sum(len(biz.frames[n]) for n in names) <= MAX_COMBINED_ROWS:
            key = f"ALL {os.path.commonprefix(names)}*"
            comb = pd.concat([biz.frames[n].assign(**{SOURCE_COL: n}) for n in names], ignore_index=True)
            if GROUP_COL in comb:   # record numbers restart in every table
                comb[GROUP_COL] = comb.groupby([SOURCE_COL, GROUP_COL], sort=False, dropna=False).ngroup()
            biz.frames[key], biz.combined[key], biz.roles[key] = comb, names, biz.roles.get(names[0], {})
            biz.parsed[key] = [_with_range(p, comb) for p in biz.parsed[names[0]]]
            if names[0] in biz.filled:
                biz.filled[key] = biz.filled[names[0]]
    return biz


def _with_range(p: Parsed, df: pd.DataFrame) -> Parsed:
    if p.kind != "date" or not df[p.col].notna().any():
        return p
    return Parsed(p.raw, p.col, p.kind, p.share, str(df[p.col].min().date()), str(df[p.col].max().date()))


def cache_path(split, biz_id: str, kind: str = "biz") -> Path:
    """Keyed by the split folder's name so train / public / private never collide."""
    return CACHE / f"{kind}_v{CACHE_VERSION}_{split_root(split).name}_{biz_id}.pkl"


def _to_plain(biz: Business) -> dict:
    """Plain pandas / dict / tuple content only, so the cache does not depend on where the
    notebook classes were defined (a Jupyter kernel's __main__ or run_lib_cells' namespace)."""
    return {"biz_id": biz.biz_id, "readme": vars(biz.readme), "frames": biz.frames, "roles": biz.roles,
            "parsed": {k: [tuple(vars(p).values()) for p in ps] for k, ps in biz.parsed.items()},
            "filled": biz.filled, "combined": biz.combined}


def _from_plain(d: dict) -> Business:
    return Business(d["biz_id"], Readme(**d["readme"]), d["frames"], d["roles"],
                    {k: [Parsed(*p) for p in ps] for k, ps in d["parsed"].items()}, d["filled"], d["combined"])


def _replace_into(path: Path, write) -> Path:
    """Write through a temp file and rename, so a killed run never leaves a half-written cache behind."""
    tmp = path.with_name(f"{path.name}.{os.getpid()}.tmp")
    write(tmp)
    tmp.replace(path)
    return path


def load_business_cached(split, biz_id: str) -> Business:
    path = cache_path(split, biz_id)
    if path.exists():
        try:
            return _from_plain(pickle.loads(path.read_bytes()))
        except (EOFError, pickle.UnpicklingError, AttributeError, TypeError):
            pass   # unreadable cache (older layout or torn write): rebuild it
    biz = load_business(split_root(split) / "data" / biz_id)
    CACHE.mkdir(parents=True, exist_ok=True)
    _replace_into(path, lambda p: p.write_bytes(pickle.dumps(_to_plain(biz))))
    return biz


def sandbox_tables(split, biz: Business) -> Path:
    """The pickle the code sandbox loads: plain DataFrames + README text."""
    path = cache_path(split, biz.biz_id, "tables")
    return path if path.exists() else _replace_into(path, lambda p: save_tables(p, biz.frames, biz.readme.text))

## A5. Table description for the prompt
Per column: the README's meaning, cell types, blank share, distinct count, the parsed column to
use (with its date range), and ALL values of low-cardinality columns (statuses, channels, payment
methods are spelled many ways). Same-column tables are shown once, as their `ALL …*` view.

In [ ]:
MAX_DISTINCT_LISTED = 20
PROFILE_SAMPLE = 3000


def _fmt(v) -> str:
    if isinstance(v, dt.datetime):   # includes pd.Timestamp; datetime(...) keeps it apart from str
        return f"datetime({v.isoformat(sep=' ')})"
    s = repr(v)
    return s if len(s) <= 40 else s[:37] + "...'"


def _parsed_note(ps: list[Parsed]) -> str:
    notes = []
    for p in ps:
        if p.kind == "date":
            notes.append(f"use {p.col!r} (datetime64, {p.share:.0%} read, {p.lo}..{p.hi})")
        elif p.kind == "datetime":
            notes.append(f"{p.col!r} keeps the time")
        else:
            notes.append(f"use {p.col!r} (float, {p.share:.0%} read)")
    return " → " + "; ".join(notes) if notes else ""


def describe_frame(name: str, biz: Business, n_examples: int = 4) -> str:
    df, roles = biz.frames[name], biz.roles.get(name, {})
    parsed = biz.parsed.get(name, [])
    by_raw: dict[str, list[Parsed]] = {}
    for p in parsed:
        by_raw.setdefault(p.raw, []).append(p)
    derived = {p.col for p in parsed} | {SOURCE_COL, GROUP_COL}
    raw_cols = [c for c in df.columns if c not in derived]
    lines = [f"### tables[{name!r}]  ({len(df)} rows x {len(raw_cols)} raw columns)"]
    if name in biz.combined:
        parts = biz.combined[name]
        lines.append(f"  = the {len(parts)} tables {parts[0]!r} … {parts[-1]!r} stacked; {SOURCE_COL!r} names the source")
    if biz.filled.get(name) == GROUP_COL:
        lines.append(f"  a line whose date is blank continues the record above it: {GROUP_COL!r} numbers the records; "
                     "dates and names are filled down onto every line of a record, amounts stay on the line that has them")
    elif name in biz.filled:
        lines.append(f"  order-level cells already filled down onto each order's lines (order id {biz.filled[name]!r})")
    for c in raw_cols:
        col = df[c]
        nonnull = col.dropna()
        sample = nonnull.head(PROFILE_SAMPLE).tolist()
        kinds = sorted({type(v).__name__ for v in sample})
        nunique = nonnull.nunique()
        blank = f", {col.isna().mean():.0%} blank" if col.isna().any() else ""
        role = f" — README: {roles[str(c)]}" if str(c) in roles else ""
        head = f"- {c!r}{role} [{'/'.join(kinds) or 'empty'}{blank}, {nunique} distinct]"
        note = _parsed_note(by_raw.get(str(c), []))
        if 0 < nunique <= MAX_DISTINCT_LISTED and any(isinstance(v, str) for v in sample):
            vc = nonnull.astype(str).value_counts()
            lines.append(head + note + ": ALL values " + ", ".join(f"{_fmt(k)}×{n}" for k, n in vc.items()))
        else:
            ex = list(dict.fromkeys(sample))[:n_examples]
            lines.append(head + note + ": " + ", ".join(_fmt(v) for v in ex))
    return "\n".join(lines)


def main_keys(biz: Business) -> list[str]:
    """Combined views first, then tables that are not part of one."""
    inside = {n for names in biz.combined.values() for n in names}
    return list(biz.combined) + [n for n in biz.frames if n not in inside and n not in biz.combined]


def describe_business(biz: Business) -> str:
    keys = main_keys(biz)
    n_parts = sum(len(v) for v in biz.combined.values())
    parts = [f"## README.md of {biz.biz_id}\n", biz.readme.text.strip(), "\n## Loaded tables",
             "Keys of `tables` to use: " + ", ".join(repr(n) for n in keys)
             + (f" (the {n_parts} single tables inside the ALL views also exist)" if n_parts else "")]
    parts += [describe_frame(n, biz) for n in keys]
    return "\n".join(parts)

# Part B — EDA
## B0. Load everything (raw tables cached: openpyxl needs a few minutes for all shops)

In [ ]:
import matplotlib.pyplot as plt
import unicodedata

pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_colwidth", 80)


def has_diacritics(s: str) -> bool:
    plain = "".join(c for c in unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
                    if unicodedata.category(c) != "Mn")
    return plain != s


CELL_PATTERNS = {
    "int": r"^-?\d+$", "decimal": r"^-?\d+\.\d+$", "thousands_dot": r"^-?\d{1,3}(\.\d{3})+$",
    "thousands_comma": r"^-?\d{1,3}(,\d{3})+$", "money_dong": r"^-?[\d.,]+\s?(đ|đồng|vnđ|vnd)$",
    "money_k": r"^-?\d+([.,]\d+)?\s?k$", "money_tr": r"^-?\d+([.,]\d+)?\s?tr(\d+)?$",
    "money_word": r"^-?\d+([.,]\d+)?\s?(triệu|nghìn|ngàn|tỷ)$", "num_with_unit": r"^\d+([.,]\d+)?\s+\D+$",
    "date_dmy": r"^\d{1,2}/\d{1,2}/\d{4}$", "date_dmy_dash": r"^\d{1,2}-\d{1,2}-\d{4}$", "date_dm": r"^\d{1,2}/\d{1,2}$",
    "date_iso": r"^\d{4}-\d{2}-\d{2}([ T]\d{2}:\d{2}(:\d{2})?)?$", "date_thg": r"^\d{1,2} thg \d{1,2}( \d{4})?$",
    "time_hm": r"^\d{1,2}:\d{2}(:\d{2})?$", "bool_word": r"^(true|false|có|không|x)$",
}
_CELL_RX = {k: re.compile(v, re.I) for k, v in CELL_PATTERNS.items()}


def cell_kind(v) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return "empty"
    if isinstance(v, (pd.Timestamp, dt.date)):
        return "datetime_obj"
    if isinstance(v, bool):
        return "bool_obj"
    if isinstance(v, (int, float)):
        return "number_obj"
    s = str(v).strip()
    return next((k for k, rx in _CELL_RX.items() if rx.match(s)), "text") if s else "empty"


def role_columns(readme_cols: dict[str, str], role_prefix: str) -> list[str]:
    return [c for c, r in readme_cols.items() if r.startswith(role_prefix)]


def split_dir(split):
    return split_root(split)


def load_split_tables(split):
    """Raw tables of every shop, cached as plain (file, sheet, DataFrame) tuples."""
    path = CACHE / f"raw_v{CACHE_VERSION}_{split}.pkl"
    if not path.exists():
        tabs = {b: [(t.file, t.sheet, t.df) for t in read_raw_tables(split_root(split) / "data" / b)]
                for b in business_ids(split)}
        CACHE.mkdir(parents=True, exist_ok=True)
        path.write_bytes(pickle.dumps(tabs))
    return {b: [RawTable(*t) for t in ts] for b, ts in pickle.loads(path.read_bytes()).items()}


TABLES = {s: load_split_tables(s) for s in SPLITS}
READMES = {s: {b: parse_readme(split_root(s) / "data" / b / "README.md") for b in business_ids(s)} for s in SPLITS}
Q = pd.concat([load_questions(s).assign(split=s) for s in SPLITS], ignore_index=True)
A = load_answers("train")
QA = Q[Q.split == "train"].merge(A, on="id")
print({s: len(TABLES[s]) for s in SPLITS}, "businesses;", len(Q), "questions")

# ## B1. Câu hỏi: ngôn ngữ

In [ ]:
EMOJI = re.compile("[\U0001F300-\U0001FAFF☀-➿]")
TEEN = re.compile(r"\b(dthu|bn|bnhiu|bnh|nhiu|nhiêu|z|zậy|ko|k|hok|mik|mk|kua|kó|cko|jùm|nhứt|th\d|t\d+|đc|dc|sp|ad|e ơi|o đờ|bill)\b", re.I)

Q["len"] = Q.question.str.len()
Q["no_diacritics"] = ~Q.question.map(has_diacritics)
Q["emoji"] = Q.question.str.contains(EMOJI)
Q["teencode"] = Q.question.str.contains(TEEN)
Q.groupby("split")[["len", "no_diacritics", "emoji", "teencode"]].mean().round(3)

In [ ]:
for s in SPLITS:
    print(f"--- {s}: 8 câu ngẫu nhiên")
    print(Q[Q.split == s].sample(8, random_state=0)[["business_id", "as_of_date", "question"]].to_string(index=False))

## B2. Chủ đề và biểu thức thời gian
Gắn nhãn thô bằng regex trên câu đã bỏ dấu (`fold`). Một câu có thể có nhiều nhãn.
Đây là công cụ EDA, không phải bộ phân loại để dùng khi chạy thật.

In [ ]:
TOPIC = {
    "revenue": r"doanh thu|dthu|doanh so|tien ban|tien hang|thu ve|ban (duoc |dc )?(bao nhieu|bn|bnhiu|nhieu|may) tien|nhieu tien nhat",
    "orders": r"\bdon\b|o do|\bbill\b|hoa don",
    "cancel": r"\bhuy\b",
    "return": r"tra hang|tra lai|doi tra",
    "debt": r"\bno\b|cong no|chua tra|thieu tien|tra gop|gop",
    "inventory": r"\bton\b|het hang|con hang|con lai",
    "purchase": r"\bnhap\b|nha cung cap|\bncc\b",
    "profit_cost": r"\blai\b|\bloi\b|gia von|chi phi",
    "price": r"\bgia\b",
    "customer": r"\bkhach\b",
    "staff": r"nhan vien|\btho\b|\bnv\b",
    "item_rank": r"mat hang|san pham|\bsp\b|\bmon\b|\bhang\b",
    "payment": r"tien mat|chuyen khoan|\bcod\b|vi dien tu|hinh thuc|kenh|zalo|shopee|tiktok|grab",
    "expiry_batch": r"han (dung|su dung)|het han|\blo\b",
    "delivery": r"giao|cho di|\bdot\b",
    "warranty": r"bao hanh",
    "domain_specific": r"topping|size|\(m\)|\(l\)|voucher|diem|bim|sua bot|hoat chat|ke don|bien so|luot xe|imei|may cu|dong may|hoa hong",
}
SHAPE = {
    "pct": r"%|phan tram|ti le|ty le|ti trong|ty trong|chiem",
    "compare": r"so voi|hon hay kem|tang hay giam|len hay xuong|chenh|cung ky|thay doi",
    "rank": r"nhat\b|nhứt|\btop\s?\d|xep|cao xuong thap|liet ke",
    "count_cond": r"(tu|it nhat|hon|tren|qua|duoi)\s?\d",
}
TIME = {
    "today": r"hom nay|hnay|ngay nay",
    "yesterday": r"hom qua",
    "this_week": r"tuan nay",
    "last_week": r"tuan (truoc|roi|vua roi)",
    "this_month": r"thang nay|\bth nay",
    "last_month": r"thang (truoc|roi)",
    "last_n_days": r"\d+ ngay (gan|qua|vua)",
    "ytd": r"tu dau nam|dau nam (toi|den)",
    "quarter": r"\bquy\b|\bq[1-4]\b",
    "tet": r"\btet\b",
    "same_period_ly": r"cung ky",
    "explicit_month": r"(thang|\bth|\bt)\s?\d{1,2}\s?(/|nam|-)\s?\d{2,4}|\b\d{1,2}/\d{2,4}\b(?!/)",
    "explicit_date": r"\b\d{1,2}/\d{1,2}/\d{2,4}\b|ngay \d{1,2}",
    "until_date": r"(den|toi) het (ngay|hom)|tinh (den|toi)",
    "shift": r"ca (sang|chieu)",
    "weekday": r"thu (hai|ba|tu|nam|sau|bay|2|3|4|5|6|7)|chu nhat",
}


def tag(texts, table):
    f = texts.map(fold)
    return pd.DataFrame({k: f.str.contains(v, regex=True) for k, v in table.items()})


Qf = Q.question
tags = pd.concat({"topic": tag(Qf, TOPIC), "shape": tag(Qf, SHAPE), "time": tag(Qf, TIME)}, axis=1)
by_split = tags.groupby(Q.split).mean().T.round(3)
by_split["diff_pub_minus_train"] = (by_split.public_test - by_split.train).round(3)
by_split.sort_values("diff_pub_minus_train")

In [ ]:
print("Câu không bắt được biểu thức thời gian nào:")
no_time = ~tags["time"].any(axis=1)
print(no_time.groupby(Q.split).mean().round(3).to_dict())
print(Q[no_time].sample(12, random_state=1)[["split", "question"]].to_string(index=False))

### Type đáp án theo shape (train)

In [ ]:
tr_tags = tags[Q.split == "train"].set_index(Q[Q.split == "train"].id)
qa_t = QA.set_index("id").join(tr_tags["shape"])
pd.crosstab(qa_t["type"], [qa_t["pct"], qa_t["rank"]], margins=True)

## B3. Đáp án (train)

In [ ]:
print(A.type.value_counts().to_frame("n").assign(share=lambda d: (d.n / d.n.sum()).round(3)).to_string())
per_biz = pd.crosstab(QA.business_id, QA.type)
per_biz.describe().round(1)

In [ ]:
num = QA[QA.type == "number"].answer.astype(float)
print("number: integer share", (num % 1 == 0).mean().round(3), "| negative", (num < 0).mean().round(3),
      "| median", num.median(), "| >=1e6 share", (num.abs() >= 1e6).mean().round(3))
pct = QA[QA.type == "percent"].answer.astype(float)
print("percent: negative share", (pct < 0).mean().round(3), "| >100 share", (pct.abs() > 100).mean().round(3))
date_answers = QA[QA.type == "date"].answer   # not `dates`: that name is the parse helper used by load_business
print("date: month-only share", date_answers.str.len().eq(7).mean().round(3))
lst = QA[QA.type == "list"].answer
print("list length:", Counter(map(len, lst)).most_common(), "| numeric lists",
      round(lst.map(lambda x: all(isinstance(v, (int, float)) for v in x)).mean(), 3))
print("alias non-empty by type:", QA[QA.alias.map(len) > 0].type.value_counts().to_dict())

In [ ]:
print("Câu unanswerable trong train:")
print(QA[QA.type == "unanswerable"][["business_id", "as_of_date", "question"]].to_string(index=False))

In [ ]:
print("Table answers:")
for _, r in QA[QA.type == "table"].head(4).iterrows():
    print(r.question, "->", r.answer["columns"], r.answer["rows"][:2])

## B4. Shop: ngành, file, kích thước
Ngành đoán từ từ khoá trong README cộng câu hỏi của shop, chỉ để nhóm shop khi so sánh.
Thứ tự quan trọng: "điện thoại" phải đứng sau vì cột "số điện thoại" và kênh "qua điện thoại" có ở mọi ngành.

In [ ]:
INDUSTRY = [
    ("nhà thuốc", r"hoat chat|thuoc ke don"),
    ("sửa xe", r"sua xe|luot xe"),
    ("mẹ và bé", r"me va be|\bbim\b|sua bot"),
    ("điện thoại", r"imei|dong may|tra gop|may cu|cua hang dien thoai"),
    ("VLXD", r"xi mang|dot giao|giao nhieu dot"),
    ("quán ăn/nhậu", r"phong lanh|khu vuc cua ban|so khach cua don"),
    ("cà phê/trà sữa", r"topping|size [ml]\b|\((m|l)\)"),
]


def industry(text):
    f = fold(text)
    return next((name for name, rx in INDUSTRY if re.search(rx, f)), "bán lẻ/tạp hoá")


rows = []
for s in SPLITS:
    for b, tabs in TABLES[s].items():
        rd = READMES[s][b]
        qtext = " ".join(Q[Q.business_id == b].question)
        rows.append(dict(split=s, biz=b, industry=industry(rd.text + " " + qtext),
                         col_docs=len(rd.columns), files=len({t.file for t in tabs}),
                         tables=len(tabs), rows=sum(len(t.df) for t in tabs),
                         formats=",".join(sorted({t.file.rsplit(".", 1)[-1] for t in tabs})),
                         last_date=rd.last_date, n_q=(Q.business_id == b).sum(),
                         kind=rd.text.split("\n")[2][:60]))
BIZ = pd.DataFrame(rows)
print("Shop có README không mô tả cột nào (chỉ có Quy ước):",
      BIZ[BIZ.col_docs == 0].groupby("split").biz.apply(list).to_dict())
pd.crosstab(BIZ.industry, BIZ.split)

**Lưu ý:** nhãn ngành ở trên chỉ gần đúng (vd. dev_004 là VLXD nhưng regex gán "điện thoại").
Đã kiểm tay bằng README: train **không** có shop điện thoại, sửa xe, mẹ và bé (rulebook train chỉ có mục
Cà phê, Nhà thuốc, VLXD). Public có 4 shop ngành mới: dev_011, dev_014 (điện thoại), dev_012 (sửa xe),
dev_013 (mẹ và bé) = 210/1000 câu.

In [ ]:
print(BIZ.sort_values(["split", "industry"]).to_string(index=False))

## B5. Header và độ phủ README
Với mỗi bảng: tìm dòng header khớp nhiều tên cột README nhất trong 10 dòng đầu.

In [ ]:
hrows = []
for s in SPLITS:
    for b, tabs in TABLES[s].items():
        for t in tabs:
            cols = readme_columns_for(READMES[s][b], t)
            row, cov = find_header_row(t.df, list(cols))
            hrows.append(dict(split=s, biz=b, file=t.file, sheet=t.sheet, n_rows=len(t.df), n_cols=t.df.shape[1],
                              readme_cols=len(cols), header_row=row, coverage=cov))
HDR = pd.DataFrame(hrows)
# Mô tả khớp tên file nhưng không khớp cột nào (vd. sheet phụ `Ghi tắt` của dev_011) thì coi như không có mô tả.
HDR.loc[HDR.coverage == 0, "readme_cols"] = 0
print("Bảng không có mô tả cột trong README:", (HDR.readme_cols == 0).sum(), "/", len(HDR))
print("Phân bố dòng header:", HDR[HDR.readme_cols > 0].header_row.value_counts().to_dict())
print("Độ phủ cột README (bảng có mô tả):")
print(HDR[HDR.readme_cols > 0].coverage.describe().round(3).to_string())
HDR[(HDR.readme_cols > 0) & (HDR.coverage < 1)].head(20)

In [ ]:
print("Bảng README không mô tả (thường là sheet phụ):")
print(HDR[HDR.readme_cols == 0][["split", "biz", "file", "sheet", "n_rows", "n_cols"]].head(25).to_string(index=False))

## B6. Độ bẩn giá trị ô
Đếm kiểu giá trị (tiền `55k`, `2,825tr`, `55.000đ`, số kèm đơn vị, ngày `1/4`, ...) trên mọi ô dưới header.

In [ ]:
krows = []
for s in SPLITS:
    for b, tabs in TABLES[s].items():
        hdr = HDR[(HDR.split == s) & (HDR.biz == b)]
        cnt = Counter()
        for t, (_, h) in zip(tabs, hdr.iterrows()):
            body = t.df.iloc[h.header_row + 1:]
            for col in body.columns:
                cnt.update(body[col].map(cell_kind).tolist())
        krows.append(dict(split=s, biz=b, **cnt))
KIND = pd.DataFrame(krows).fillna(0).set_index(["split", "biz"])
MESSY = ["money_dong", "money_k", "money_tr", "money_word", "thousands_dot", "thousands_comma", "num_with_unit",
         "date_dmy", "date_dmy_dash", "date_dm", "date_thg", "date_iso", "datetime_obj", "time_hm"]
messy = (KIND.reindex(columns=MESSY, fill_value=0) > 0).astype(int)
print("Số shop có ít nhất một ô kiểu đó:")
print(messy.groupby(level="split").sum().T.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(messy.values, aspect="auto", cmap="Greys")
ax.set_xticks(range(len(MESSY)), MESSY, rotation=60, ha="right")
ax.set_yticks(range(len(messy)), [b for _, b in messy.index], fontsize=7)
ax.set_title("Kiểu giá trị xuất hiện ở mỗi shop")
plt.tight_layout()

In [ ]:
def sample_cells(split, biz, kind, k=8):
    hdr = HDR[(HDR.split == split) & (HDR.biz == biz)]
    out = []
    for t, (_, h) in zip(TABLES[split][biz], hdr.iterrows()):
        body = t.df.iloc[h.header_row + 1:]
        for col in body.columns:
            vals = body[col][body[col].map(cell_kind) == kind]
            out += vals.astype(str).head(3).tolist()
    return out[:k]


for kind in ["money_k", "money_tr", "money_dong", "num_with_unit", "date_dm", "date_thg"]:
    ex = next(((s, b) for (s, b), v in messy[kind].items() if v), None)
    if ex:
        print(f"{kind:14s} {ex[1]}: {sample_cells(*ex, kind)}")

## B7. Trạng thái đơn, khách lẻ, ô trống theo nhóm, dòng trùng

In [ ]:
def framed(split, biz):
    """(table, header-applied df, README column map) for every described table of a shop."""
    hdr = HDR[(HDR.split == split) & (HDR.biz == biz)]
    for t, (_, h) in zip(TABLES[split][biz], hdr.iterrows()):
        cols = readme_columns_for(READMES[split][biz], t)
        if cols:
            yield t, with_header(t.df, h.header_row), cols


status, walkin, frows = Counter(), Counter(), []
for s in SPLITS:
    for b in TABLES[s]:
        for t, df, cols in framed(s, b):
            for c in role_columns(cols, "trạng thái đơn"):
                if c in df:
                    status.update((s, fold(v)) for v in df[c].dropna().astype(str))
            for c in role_columns(cols, "tên khách"):
                if c in df:
                    vc = df[c].dropna().astype(str).map(fold).value_counts()
                    walkin.update({(s, k): v for k, v in vc.head(3).items() if len(k) <= 12})
            for c in role_columns(cols, "mã đơn"):
                if c in df and len(df) > 0:
                    blank = df[c].isna().mean()
                    dup = df.duplicated(keep="first").mean()
                    frows.append(dict(split=s, biz=b, file=t.file, sheet=t.sheet, rows=len(df),
                                      order_id_blank=round(blank, 3), exact_dup_rows=round(dup, 4)))

st = pd.Series(status).unstack(0).fillna(0).astype(int).sort_values("train", ascending=False)
print("Giá trị trạng thái đơn (đã bỏ dấu):", len(st))
print(st.to_string())

In [ ]:
print("Tên khách xuất hiện nhiều nhất và ngắn (ứng viên 'khách lẻ'):")
print(pd.Series(walkin).unstack(0).fillna(0).astype(int).sort_values("train", ascending=False).head(25).to_string())

In [ ]:
FILL = pd.DataFrame(frows)
print("Bảng có ô mã đơn để trống (ghi ở dòng đầu đơn):", (FILL.order_id_blank > 0).sum(), "/", len(FILL))
print("Bảng có dòng trùng hệt nhau:", (FILL.exact_dup_rows > 0).sum(), "/", len(FILL))
FILL.sort_values("order_id_blank", ascending=False).head(15)

## B8. Rulebook: train vs public

In [ ]:
def rulebook_rules(split):
    text = (split_dir(split) / "rulebook.md").read_text(encoding="utf-8")
    rules, section = {}, None
    for line in text.splitlines():
        if line.startswith("#"):
            section = line.lstrip("# ").strip()
        m = re.match(r"^\| ([^|]+?) \| (.+) \|$", line)
        if m and not set(m.group(1)) <= set("-") and m.group(1) not in ("Luật", "Cách nói", "Cách ghi", "Từ", "type"):
            rules[(section, m.group(1))] = m.group(2)
    return rules


R = {s: rulebook_rules(s) for s in SPLITS}
added = sorted(set(R["public_test"]) - set(R["train"]))
removed = sorted(set(R["train"]) - set(R["public_test"]))
changed = sorted(k for k in set(R["train"]) & set(R["public_test"]) if R["train"][k] != R["public_test"][k])
print(f"rules: train={len(R['train'])} public={len(R['public_test'])} | added={len(added)} removed={len(removed)} changed={len(changed)}")
for name, keys in [("ADDED", added), ("REMOVED", removed), ("CHANGED", changed)]:
    print(f"\n## {name}")
    for sec, rule in keys:
        print(f"- [{sec}] {rule}")

## B9. `as_of_date` so với khoảng dữ liệu

In [ ]:
last = {b: pd.to_datetime(r.last_date, dayfirst=True) for s in SPLITS for b, r in READMES[s].items()}
Q["as_of"] = pd.to_datetime(Q.as_of_date)
Q["days_before_end"] = (Q.business_id.map(last) - Q.as_of).dt.days
print(Q.groupby("split").days_before_end.describe().round(1).to_string())
print("as_of == ngày cuối dữ liệu:", Q.groupby("split").days_before_end.apply(lambda d: (d == 0).mean().round(3)).to_dict())
print("as_of sau ngày cuối dữ liệu:", Q.groupby("split").days_before_end.apply(lambda d: (d < 0).mean().round(3)).to_dict())

In [ ]:
print("Số as_of_date khác nhau mỗi shop:", Q.groupby(["split", "business_id"]).as_of_date.nunique().groupby("split").describe().round(1).to_string())

## B10. Data card per shop (normalized data)
What each shop records, from README meanings and column names: the entities a question can need.
A question about an entity the shop does not record is unanswerable (rulebook §7). Date range is
taken from the parsed sale-date columns, so it is exact (EDA §10 of the first pass used a heuristic).

In [ ]:
ENTITIES = {
    "order lines": r"ma don|so hoa don|so thu tu don",
    "customers": r"ma khach|ten khach|nhom khach|khach hang",
    "customer groups": r"nhom khach",
    "products / catalog": r"ma hang|ma sku|nhom hang|danh muc|don vi co ban",
    "unit conversion": r"quy doi|he so|don vi lon",
    "purchases": r"phieu nhap|ngay nhap|nha cung cap|gia nhap",
    "payments / debt": r"con no|da thanh toan|so tien tra|tien tra|khach tra|cong no|han thanh toan|phieu thu",
    "payment method": r"hinh thuc thanh toan|phuong thuc",
    "sales channel": r"kenh ban|kenh",
    "inventory": r"\bton\b|ton kho|ton cuoi|the kho",
    "batches / expiry": r"\blo\b|so lo|han su dung|han dung",
    "price history": r"gia moi|ap dung|gia cu|lich su gia",
    "cost": r"gia von",
    "delivery": r"ngay giao|dot giao|tai xe|giao hang",
    "staff": r"nhan vien|\btho\b|thu ngan",
    "expenses": r"chi phi|khoan chi",
    "returns": r"tra hang|tra lai|so luong tra|hoan",
    "warranty": r"bao hanh",
}
SALE_DATE_ROLE = re.compile(r"^(ngay ban|thoi diem dat don|ngay dat|ngay sua|ngay hoa don)")
DATE_NAME = re.compile(r"ngay|thoi gian|thoi diem|date")
NOT_SALE_DATE = re.compile(r"giao|han|sinh|tao|dang ky|nhap|vao lam|ap dung")


def sale_date_columns(biz) -> list[Parsed]:
    """Parsed sale-date columns: by README meaning; for undocumented shops, date-named columns of
    tables that have an order id."""
    out = []
    for k in main_keys(biz):
        roles = biz.roles.get(k, {})
        for p in biz.parsed.get(k, []):
            if p.kind != "date":
                continue
            if roles:
                if SALE_DATE_ROLE.match(fold(roles.get(p.raw, ""))):
                    out.append(p)
            elif order_id_column(biz.frames[k], {}) and DATE_NAME.search(fold(p.raw)) \
                    and not NOT_SALE_DATE.search(fold(p.raw)):
                out.append(p)
    return out


def data_card(split, biz_id):
    biz = load_business_cached(split, biz_id)
    # Column names and their README meanings only: README free text lists products ('Sữa đặc Ông Thọ'
    # would read as 'thợ', staff).
    text = fold(" ".join(f"{k} {' '.join(map(str, df.columns))} {' '.join(biz.roles.get(k, {}).values())}"
                         for k, df in biz.frames.items()))
    described = sum(bool(r) for r in biz.roles.values())
    sale = sale_date_columns(biz)
    los, his = [p.lo for p in sale if p.lo], [p.hi for p in sale if p.hi]
    card = dict(split=split, biz=biz_id, tables=len(main_keys(biz)), combined=len(biz.combined),
                rows=sum(len(biz.frames[k]) for k in main_keys(biz)),
                readme=("full" if described >= 0.6 * len(biz.roles) else "partial" if described else "none"),
                first_sale=min(los) if los else "", last_sale=max(his) if his else "",
                readme_last=biz.readme.last_date)
    card.update({e: bool(re.search(rx, text)) for e, rx in ENTITIES.items()})
    return card


CARDS = pd.DataFrame([data_card(s, b) for s in SPLITS for b in business_ids(s)])
print(CARDS.groupby("split").readme.value_counts().to_string())
CARDS.drop(columns=list(ENTITIES))

In [ ]:
ent = CARDS.groupby("split")[list(ENTITIES)].mean().T.round(2)
ent["train_minus_public"] = (ent["train"] - ent["public_test"]).round(2)
ent.sort_values("train_minus_public")

In [ ]:
print("Shops by entity (✓ recorded):")
CARDS.set_index("biz")[list(ENTITIES)].replace({True: "✓", False: ""})